## Evaluation Criteria

### EvalSet - collection of one or more test cases.
- Each test case specifies the inputs that will be sent to the agent during the test run. 

EvalSet is a rich data structure that can include :
- user’s query or prompt
- list of the tool calls we expect the agent to make in a specific order to correctly respond to the user’s query.
- Expected intermediate agent responses
- Final response

In [ ]:
# Evalset.json :
{
  "eval_set_id": "generic_eval_set_id",
  "name": "Generic Evaluation Set Name",
  "description": "A description of what this evaluation set is for.",
  "eval_cases": [
    {
      "eval_id": "generic_eval_case_id",
      "conversation": [
        {
          "invocation_id": "unique_invocation_identifier",
          "user_content": { // The user content for this turn
            "parts": [
              {
                "text": "The user's initial prompt for this turn."
              }
            ],
            "role": "user"
          },
          "final_response": { // The expected final response
            "parts": [
              {
                "text": "The expected 'golden' final text response from the agent."
              }
            ],
            "role": "model"
          },
          "intermediate_data": {
            "tool_uses": [ // The expected intermediate tool use trajectory
              {
                "name": "expected_tool_name_1",
                "args": {
                  "parameter_name": "parameter_value"
                }
              },
              {
                "name": "expected_tool_name_2",
                "args": {}
              }
            ],
            "intermediate_responses": [] // Holds expected intermediate agent responses
          }
        }
      ],
      "session_input": {
        "app_name": "your_app_name",
        "user_id": "your_user_id"
      }
    }
  ]
}


- The ADK framework uses a specific naming convention for these files: <eval_set_id>.evalset.json. The file name must match the eval_set_id defined inside the JSON file, as this allows the adk eval command to correctly identify and load the test suite. 

### EvalConfig -  specifies one or more evaluation criteria that the ADK will use to score the agent’s behavior against the test cases in the EvalSet.

### (A) - Process and Reasoning criterion

#### 1. Tool trajectory average score

*  Tool trajectory average score : is essential for verifying that the agent is following the correct plan. It compares the actual sequence of tools called by the agent against a list of expected calls that we provide in our EvalSet.

In [ ]:
{
  "criteria": {
    "tool_trajectory_avg_score": {
      "threshold": 1.0,
      "match_type": 2
    }
  }
}

It can be configured with one of three match_type options: :
- EXACT or 0: The agent’s tool calls must be a perfect, one-to-one match with the expected list
- IN_ORDER or 1:It requires that all expected tool calls are present in the actual list and in the same relative order, but it allows for other, unexpected tool calls to occur in between.
- ANY_ORDER or 2 : this simply checks that all expected tool calls were made, regardless of their order.

#### 2. Rubric-based tool use quality

- Rubric-based tool use quality : LLM-judged criterion for evaluating the quality of an agent’s tool usage against a set of custom rules that we define

In [ ]:
{
  "criteria": {
    "rubric_based_tool_use_quality_v1": {
      "threshold": 1.0,
      "judge_model_options": {
        "judge_model": "gemini-2.5-flash"
      },
      "rubrics": [
        {
          "rubric_id": "wikipedia_called",
          "rubric_content": {
            "text_property": "The agent calls the wikipedia tool with title extracted from the user's query."
          }
        }
      ]
    }
  }
}

- Parameter correctness: Did the agent provide valid and appropriate arguments to the tools it called?

- Workflow adherence: Did the agent follow a prescribed sequence of tool calls for a specific type of task?

- Tool selection logic: Did the agent select the most efficient and logical tool for the user’s specific request?

### (B) - Response Quality criterion

#### 1. Response match score

- response_match_score uses a lexical-matching algorithm called ROUGE-1, which measures the literal overlap of single words between the agent’s response and our reference answer.

In [ ]:
{
  "criteria": {
    "response_match_score": 0.8
  }
}

#### 2. Final response match

- The final_response_match_v2 is a more sophisticated, LLM-judged criterion. It uses an LLM to determine if the agent’s response is semantically equivalent to our reference answer

In [ ]:
{
  "criteria": {
    "final_response_match_v2": {
      "threshold": 0.8,
      "judge_model_options": {
            "judge_model": "gemini-2.5-flash"
          }
        }
    }
}


#### 3. Rubric-based final response quality

-  rubric_based_final_response_quality_v1 is the most flexible criterion for evaluating final output, as it does not require a predefined reference answer. Instead, it uses an LLM-as-a-Judge to score the agent’s response against a set of abstract quality rules, or rubrics, that we define.

In [ ]:
{
  "criteria": {
    "rubric_based_final_response_quality_v1": {
      "threshold": 0.8,
      "judge_model_options": {
        "judge_model": "gemini-2.5-flash"
      },
      "rubrics": [
        {
          "rubric_id": "conciseness",
          "rubric_content": {
            "text_property": "The agent's response is direct and to the point."
          }
        }
        }
      ]
    }
  }
}

### (C) - Criteria for factual grounding and safety

#### 1. Hallucination

- The hallucinations_v1 is an extremely powerful and important LLM-judged criterion. It assesses whether the agent’s response is factually grounded in a given context.
- we provide the EvalSet with the actual outputs from the tools the agent used

In [ ]:
{
  "criteria": {
    "hallucinations_v1": {
      "threshold": 0.8,
      "judge_model_options": {
            "judge_model": "gemini-2.5-flash",
          }
      "evaluate_intermediate_nl_responses": true
    }
  }
}

#### 2. Safety

- The safety_v1 criterion evaluates the safety and harmlessness of an agent’s response, checking for harmful content such as hate speech, harassment, or dangerous information. 

- the safety_v1 criterion delegates its task to a powerful, managed service on Google Cloud: the Vertex AI General AI Eval SDK. Because this criterion uses a Google Cloud service instead of running entirely within the local ADK framework, it requires additional configuration.

In [ ]:
{
  "criteria": {
    "safety_v1": 0.8
  }
}

## User simulation for conversational agents

- this uses a generative AI model (an LLM user simulator) to dynamically play the role of a user, generating prompts on the fly in response to the agent’s messages, all while trying to achieve a predefined goal

we guide the user simulator by defining a ConversationScenario in our EvalSet. This is a JSON object that outlines the user’s objective for the conversation. It has two key parts:

- starting_prompt: This is a fixed, predefined string that initiates the conversation. It is the only scripted part of the user’s dialogue and serves as the entry point for the test.

- conversation_plan: This is the most important part. It is not a script of what the user will say, but rather a high-level objective or set of goals that the user simulator should try to achieve during the conversation.

In [ ]:
conversation_scenarios = (
"""{
  "scenarios": [
    {
      "starting_prompt": "I need to book a flight.",
      "conversation_plan": "Book a one-way ticket to Paris for next Friday. After the agent finds flights, ask about the luggage policy before confirming."
    }
  ]
}""")